# ARIAN Phase 1 — Data Ingestion

Pulls historical & forecast weather, fire labels, and static geography for the
16-city pipeline. Writes `data/processed/master_daily.parquet` — the input
to all downstream notebooks.

**Inputs:** Open-Meteo (no key), NASA FIRMS archive CSVs in `data/raw/firms/`,
legacy `merged.csv` in `data/raw/legacy/`, Open-Elevation (no key).

**Output:** `data/processed/master_daily.parquet`

**Fallback logic:** Every fetch checks for a local cache first.
If the file already exists on disk, it is loaded directly — no API call.
API timeouts and rate-limits are caught and retried gracefully.

## §0 · Environment setup

In [1]:
# Install once. Comment out subsequent runs.
%pip install -q openmeteo-requests requests-cache retry-requests \
    pandas numpy pyarrow tqdm folium 2>/dev/null || \
!pip install -q openmeteo-requests requests-cache retry-requests \
    pandas numpy pyarrow tqdm folium

Note: you may need to restart the kernel to use updated packages.


In [2]:
# ─── ARIAN — Universal environment & path setup ───────────────────────────
# Auto-detects Google Colab vs. local. Works in both with no edits.
#
# Layout expected (created automatically if missing):
#     ARIAN_Data/
#         notebooks/        ← these notebooks live here
#         data/
#             raw/
#                 firms/    ← NASA FIRMS sensor folders
#                 legacy/   ← merged.parquet + supplementary CSVs
#             processed/    ← master_daily.parquet (built by notebook 1)
#             reference/    ← cities, static geography
#         outputs/          ← phase2/3/4 artefacts
#         models/           ← trained classifier + manifest
#
# Override the project root by setting the ARIAN_ROOT environment variable.
import os, sys
from pathlib import Path

PROJECT_NAME = "ARIAN_Data"

def _detect_project_root() -> Path:
    if os.environ.get("ARIAN_ROOT"):
        return Path(os.environ["ARIAN_ROOT"]).expanduser().resolve()

    if "google.colab" in sys.modules:                 # Colab branch
        from google.colab import drive
        if not os.path.ismount("/content/drive"):
            drive.mount("/content/drive")
        return Path(f"/content/drive/MyDrive/{PROJECT_NAME}")

    here = Path.cwd().resolve()                       # local branch
    for cand in [here, *here.parents]:
        if (cand / "data").is_dir() and (cand / "notebooks").is_dir():
            return cand
        if cand.name == PROJECT_NAME and (cand / "data").is_dir():
            return cand
    return here.parent if here.name == "notebooks" else here


ROOT      = _detect_project_root()
DATA      = ROOT / "data"
RAW       = DATA / "raw"
LEGACY    = RAW  / "legacy"
FIRMS     = RAW  / "firms"
PROCESSED = DATA / "processed"
REFERENCE = DATA / "reference"
OUTPUTS   = ROOT / "outputs"
MODELS    = ROOT / "models"

for p in (RAW, LEGACY, FIRMS, PROCESSED, REFERENCE, OUTPUTS, MODELS):
    p.mkdir(parents=True, exist_ok=True)

print(f"📁 Project root : {ROOT}")
print(f"   data/raw      : {RAW}")
print(f"   data/processed: {PROCESSED}")
print(f"   outputs       : {OUTPUTS}")
print(f"   models        : {MODELS}")


📁 Project root : /home/manheim666/Desktop/ARIAN ALL FOLDERS/ARIAN_ASIF
   data/raw      : /home/manheim666/Desktop/ARIAN ALL FOLDERS/ARIAN_ASIF/data/raw
   data/processed: /home/manheim666/Desktop/ARIAN ALL FOLDERS/ARIAN_ASIF/data/processed
   outputs       : /home/manheim666/Desktop/ARIAN ALL FOLDERS/ARIAN_ASIF/outputs
   models        : /home/manheim666/Desktop/ARIAN ALL FOLDERS/ARIAN_ASIF/models


In [3]:
# ─── Imports + Open-Meteo client ──────────────────────────────────────────
import io, re, time
from datetime import datetime, timedelta, timezone
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import requests, requests_cache
from retry_requests import retry
import openmeteo_requests
from tqdm.auto import tqdm
from pandas.errors import EmptyDataError

# HTTP cache stays LOCAL (SQLite + cloud sync = pain, even on Drive)
LOCAL_CACHE = (Path("/content") if "google.colab" in sys.modules else Path.home()) / ".arian_http_cache"
LOCAL_CACHE.mkdir(parents=True, exist_ok=True)

cache_session = requests_cache.CachedSession(str(LOCAL_CACHE / "openmeteo"), expire_after=86400)
retry_session = retry(cache_session, retries=5, backoff_factor=0.3)
om            = openmeteo_requests.Client(session=retry_session)

print(f"✅ HTTP cache: {LOCAL_CACHE}")

✅ HTTP cache: /home/manheim666/.arian_http_cache


## §1 · Target locations & run config

In [4]:
# ─── 16 Azerbaijani cities — single source of truth ───────────────────────
from typing import Dict, Tuple

CITIES: Dict[str, Tuple[float, float]] = {
    "Ganja":       (40.6828, 46.3606),
    "Baku":        (40.4093, 49.8671),
    "Zaqatala":    (41.6296, 46.6433),
    "Lankaran":    (38.7523, 48.8475),
    "Shaki":       (41.1975, 47.1694),
    "Jalilabad":   (39.2089, 48.2986),
    "Sumqayit":    (40.5897, 49.6686),
    "Mingachevir": (40.7639, 47.0595),
    "Shirvan":     (39.9317, 48.9299),
    "Quba":        (41.3611, 48.5261),
    "Khachmaz":    (41.4635, 48.8060),
    "Yevlakh":     (40.6183, 47.1500),
    "Gabala":      (40.9982, 47.8468),
    "Shamakhi":    (40.6303, 48.6414),
    "Nakhchivan":  (39.2089, 45.4122),
    "Barda":       (40.3744, 47.1266),
}

CONFIG = {
    "history_start":         "2012-01-20",
    "history_end":           datetime.now(timezone.utc).strftime("%Y-%m-%d"),
    "fire_buffer_km":        20,
    "forecast_horizon_days": 16,
    "polite_delay_s":        1.0,
}

cities_df = pd.DataFrame(
    [(name, lat, lon) for name, (lat, lon) in CITIES.items()],
    columns=["City", "Latitude", "Longitude"],
)
cities_df.to_parquet(REFERENCE / "cities.parquet", index=False)
cities_df

,City,Latitude,Longitude
0,Ganja,40.6828,46.3606
1,Baku,40.4093,49.8671
2,Zaqatala,41.6296,46.6433
3,Lankaran,38.7523,48.8475
4,Shaki,41.1975,47.1694
5,Jalilabad,39.2089,48.2986
6,Sumqayit,40.5897,49.6686
7,Mingachevir,40.7639,47.0595
8,Shirvan,39.9317,48.9299
9,Quba,41.3611,48.5261


## §2 · Historical hourly weather — Open-Meteo Archive

In [5]:
# ── Variable groups — split by model availability ─────────────────────────
# ERA5 provides atmospheric variables; ERA5-Land provides soil variables.
# The Archive API requires explicit model names (no "best_match").

ERA5_VARS = [
    "temperature_2m", "relative_humidity_2m", "precipitation",
    "wind_speed_10m", "wind_direction_10m", "surface_pressure",
    "shortwave_radiation",
]
ERA5_LAND_VARS = [
    "soil_temperature_0_to_7cm", "soil_moisture_0_to_7cm",
]
HOURLY_VARS = ERA5_VARS + ERA5_LAND_VARS

COLUMN_MAP = {
    "temperature_2m":             "Temperature_C",
    "relative_humidity_2m":       "Humidity_percent",
    "precipitation":              "Rain_mm",
    "wind_speed_10m":             "Wind_Speed_kmh",
    "wind_direction_10m":         "Wind_Direction_deg",
    "surface_pressure":           "Pressure_hPa",
    "shortwave_radiation":        "Solar_Radiation_Wm2",
    "soil_temperature_0_to_7cm":  "Soil_Temp_C",
    "soil_moisture_0_to_7cm":     "Soil_Moisture",
}
_REQUIRED_FEATURES = list(COLUMN_MAP.values())

def _slug(s: str) -> str:
    return re.sub(r"[^A-Za-z0-9_-]+", "_", s).strip("_")

def _fetch_with_retry(url: str, params: dict, max_retries: int = 10,
                      timeout_s: float = 60.0):
    """Call Open-Meteo, sleeping through minutely/hourly limits."""
    for attempt in range(1, max_retries + 1):
        try:
            return om.weather_api(url, params=params)
        except requests.exceptions.Timeout:
            print(f"      ⏳ request timed out ({timeout_s}s), attempt {attempt}/{max_retries}")
            time.sleep(min(30, 2 ** attempt))
        except requests.exceptions.ConnectionError as e:
            print(f"      ⏳ connection error, attempt {attempt}/{max_retries}")
            time.sleep(min(60, 2 ** attempt))
        except Exception as e:
            msg = str(e).lower()
            if   "minutely api request limit" in msg: wait = 65
            elif "hourly api request limit"   in msg: wait = 60 * 60 + 30
            elif "daily api request limit"    in msg:
                raise RuntimeError(
                    "Daily limit (10 000 calls) hit — re-run tomorrow. "
                    "Per-city checkpoints will resume where you left off."
                ) from e
            else:
                wait = min(60, 2 ** attempt)
            print(f"      ⏳ rate-limited ({attempt}/{max_retries}); sleeping {wait}s …")
            time.sleep(wait)
    raise RuntimeError(f"Exhausted {max_retries} retries for {url}")


def _parse_hourly(resp, var_list):
    """Parse hourly variables from an Open-Meteo response into a dict."""
    h = resp.Hourly()
    timestamps = pd.date_range(
        start=pd.to_datetime(h.Time(),    unit="s", utc=True),
        end=  pd.to_datetime(h.TimeEnd(), unit="s", utc=True),
        freq= pd.Timedelta(seconds=h.Interval()),
        inclusive="left",
    )
    data = {"Timestamp": timestamps}
    for i, var in enumerate(var_list):
        data[COLUMN_MAP[var]] = h.Variables(i).ValuesAsNumpy()
    return pd.DataFrame(data)


def _cache_is_complete(cache_path: Path) -> bool:
    """Check that a per-city cache file has ALL 9 features with actual data."""
    if not cache_path.exists():
        return False
    try:
        df = pd.read_parquet(cache_path)
        return all(c in df.columns and df[c].notna().any() for c in _REQUIRED_FEATURES)
    except Exception:
        return False


def fetch_history_one_city(name: str, lat: float, lon: float,
                           start: str, end: str) -> pd.DataFrame:
    """Fetch hourly history for one city using two API calls:
    1. ERA5 for atmospheric variables (wind, pressure, radiation, etc.)
    2. ERA5-Land for soil variables (soil temp, soil moisture)
    Results are merged on Timestamp."""
    cache_path = RAW / f"weather_hourly__{_slug(name)}.parquet"

    # ── 1. Try valid local cache ──────────────────────────────────────
    if _cache_is_complete(cache_path):
        print(f"   📦 {name}: loaded from local cache (all 9 features OK)")
        return pd.read_parquet(cache_path)
    elif cache_path.exists():
        df_old = pd.read_parquet(cache_path)
        missing = [c for c in _REQUIRED_FEATURES
                   if c not in df_old.columns or df_old[c].isna().all()]
        print(f"   🔄 {name}: cache incomplete (missing: {missing}) — deleting …")
        cache_path.unlink()

    # ── 2. API fetch — two calls for full coverage ────────────────────
    base_params = {"latitude": lat, "longitude": lon,
                   "start_date": start, "end_date": end,
                   "timezone": "UTC"}
    url = "https://archive-api.open-meteo.com/v1/archive"

    df_era5 = None
    df_land = None

    # Call 1: ERA5 — atmospheric variables
    try:
        print(f"   🌐 {name}: fetching ERA5 (atmospheric) …")
        p1 = {**base_params, "hourly": ERA5_VARS, "models": "era5"}
        resp1 = _fetch_with_retry(url, p1)[0]
        df_era5 = _parse_hourly(resp1, ERA5_VARS)
        n_ok = sum(1 for v in ERA5_VARS if df_era5[COLUMN_MAP[v]].notna().any())
        print(f"      ✅ ERA5: {n_ok}/{len(ERA5_VARS)} vars · {len(df_era5):,} rows")
    except Exception as exc:
        print(f"      ❌ ERA5 fetch failed: {exc}")

    # Call 2: ERA5-Land — soil variables
    try:
        print(f"   🌐 {name}: fetching ERA5-Land (soil) …")
        p2 = {**base_params, "hourly": ERA5_LAND_VARS, "models": "era5_land"}
        resp2 = _fetch_with_retry(url, p2)[0]
        df_land = _parse_hourly(resp2, ERA5_LAND_VARS)
        n_ok = sum(1 for v in ERA5_LAND_VARS if df_land[COLUMN_MAP[v]].notna().any())
        print(f"      ✅ ERA5-Land: {n_ok}/{len(ERA5_LAND_VARS)} vars · {len(df_land):,} rows")
    except Exception as exc:
        print(f"      ❌ ERA5-Land fetch failed: {exc}")

    time.sleep(0.5)  # polite pause between city pairs

    # Merge the two results
    if df_era5 is not None and df_land is not None:
        df = df_era5.merge(df_land, on="Timestamp", how="outer")
    elif df_era5 is not None:
        df = df_era5
        for v in ERA5_LAND_VARS:
            df[COLUMN_MAP[v]] = np.nan
    elif df_land is not None:
        df = df_land
        for v in ERA5_VARS:
            df[COLUMN_MAP[v]] = np.nan
    else:
        df = None

    if df is not None and not df.empty:
        df["City"], df["Latitude"], df["Longitude"] = name, lat, lon
        df = df.sort_values("Timestamp").reset_index(drop=True)

        # Only cache if we got substantial data
        avail = [c for c in _REQUIRED_FEATURES if c in df.columns and df[c].notna().any()]
        if len(avail) >= 7:  # cache if ≥7/9 features have data
            df.to_parquet(cache_path, index=False)
        print(f"   ✅ {name}: {len(avail)}/9 features · {len(df):,} rows")
        return df

    # ── 3. Legacy fallback (partial features only) ────────────────────
    legacy_csv = LEGACY / "merged.csv"
    if legacy_csv.exists():
        print(f"   📦 {name}: falling back to legacy merged.csv (partial features)")
        try:
            legacy = pd.read_csv(legacy_csv, low_memory=False)
            legacy["Timestamp"] = pd.to_datetime(legacy["Timestamp"])
            city_df = legacy[legacy["City"] == name].copy()
            if not city_df.empty:
                keep = ["Timestamp"] + [v for v in _REQUIRED_FEATURES if v in city_df.columns]
                df = city_df[keep].copy()
                for v in _REQUIRED_FEATURES:
                    if v not in df.columns:
                        df[v] = np.nan
                df["City"], df["Latitude"], df["Longitude"] = name, lat, lon
                return df
        except Exception as exc:
            print(f"   ⚠️  legacy CSV read failed for {name}: {exc}")

    print(f"   ❌ {name}: no data available — returning empty DataFrame")
    return pd.DataFrame()


def fetch_all_history() -> pd.DataFrame:
    frames = []
    for name, (lat, lon) in tqdm(CITIES.items(), desc="History"):
        df = fetch_history_one_city(
            name, lat, lon, CONFIG["history_start"], CONFIG["history_end"]
        )
        if not df.empty:
            frames.append(df)
        time.sleep(CONFIG["polite_delay_s"])
    if not frames:
        raise RuntimeError("No weather data loaded for any city. "
                           "Check API / local cache / legacy merged.csv.")
    return pd.concat(frames, ignore_index=True)


weather_hist = (fetch_all_history()
                .drop_duplicates(["City", "Timestamp"])
                .sort_values(["City", "Timestamp"])
                .reset_index(drop=True))
weather_hist.to_parquet(RAW / "weather_hourly.parquet", index=False)

print(f"\n✅ Historical weather: {len(weather_hist):,} rows · "
      f"{weather_hist['City'].nunique()} cities · "
      f"{weather_hist['Timestamp'].min()} → {weather_hist['Timestamp'].max()}")

print("\nFeature completeness:")
for c in _REQUIRED_FEATURES:
    nn = weather_hist[c].notna().sum()
    pct = nn / len(weather_hist) * 100
    status = "✅" if pct > 50 else "⚠️ "
    print(f"  {status} {c:30s} {nn:>10,} rows ({pct:.1f}%)")

weather_hist.head()

History:   0%|          | 0/16 [00:00<?, ?it/s]

   🌐 Ganja: fetching ERA5 (atmospheric) …
      ❌ ERA5 fetch failed: Daily limit (10 000 calls) hit — re-run tomorrow. Per-city checkpoints will resume where you left off.
   🌐 Ganja: fetching ERA5-Land (soil) …
      ❌ ERA5-Land fetch failed: Daily limit (10 000 calls) hit — re-run tomorrow. Per-city checkpoints will resume where you left off.
   📦 Ganja: falling back to legacy merged.csv (partial features)
   🌐 Baku: fetching ERA5 (atmospheric) …
      ❌ ERA5 fetch failed: Daily limit (10 000 calls) hit — re-run tomorrow. Per-city checkpoints will resume where you left off.
   🌐 Baku: fetching ERA5-Land (soil) …
      ❌ ERA5-Land fetch failed: Daily limit (10 000 calls) hit — re-run tomorrow. Per-city checkpoints will resume where you left off.
   📦 Baku: falling back to legacy merged.csv (partial features)
   🌐 Zaqatala: fetching ERA5 (atmospheric) …
      ❌ ERA5 fetch failed: Daily limit (10 000 calls) hit — re-run tomorrow. Per-city checkpoints will resume where you left off.
   🌐

,Timestamp,Temperature_C,Humidity_percent,Rain_mm,Wind_Speed_kmh,Wind_Direction_deg,Pressure_hPa,Solar_Radiation_Wm2,Soil_Temp_C,Soil_Moisture,City,Latitude,Longitude
0,2012-01-19 20:00:00,4.024,83.45899,0.0,35.654540,NaN,NaN,NaN,NaN,NaN,Baku,40.4093,49.8671
1,2012-01-19 21:00:00,4.024,81.38923,0.0,37.842830,NaN,NaN,NaN,NaN,NaN,Baku,40.4093,49.8671
2,2012-01-19 22:00:00,3.924,81.96262,0.2,38.941616,NaN,NaN,NaN,NaN,NaN,Baku,40.4093,49.8671
3,2012-01-19 23:00:00,3.874,81.07652,0.2,40.603470,NaN,NaN,NaN,NaN,NaN,Baku,40.4093,49.8671
4,2012-01-20 00:00:00,3.924,80.21263,0.2,41.904060,NaN,NaN,NaN,NaN,NaN,Baku,40.4093,49.8671


## §3 · Live forecast weather — Open-Meteo Forecast

In [6]:
FORECAST_VARS = HOURLY_VARS
FORECAST_CACHE = RAW / "weather_forecast.parquet"

def fetch_forecast_one_city(name: str, lat: float, lon: float,
                            forecast_days: int = 16) -> pd.DataFrame:
    params = {"latitude": lat, "longitude": lon,
              "hourly": FORECAST_VARS,
              "forecast_days": forecast_days, "timezone": "UTC"}
    try:
        resp = _fetch_with_retry("https://api.open-meteo.com/v1/forecast", params)[0]
    except Exception as exc:
        print(f"   ❌ Forecast API failed for {name}: {exc}")
        return pd.DataFrame()

    h = resp.Hourly()
    timestamps = pd.date_range(
        start=pd.to_datetime(h.Time(),    unit="s", utc=True),
        end=  pd.to_datetime(h.TimeEnd(), unit="s", utc=True),
        freq= pd.Timedelta(seconds=h.Interval()),
        inclusive="left",
    )
    data = {"Timestamp": timestamps}
    for i, var in enumerate(FORECAST_VARS):
        data[COLUMN_MAP[var]] = h.Variables(i).ValuesAsNumpy()
    df = pd.DataFrame(data)
    df["City"], df["Latitude"], df["Longitude"] = name, lat, lon
    return df


def fetch_all_forecasts(forecast_days: int = 16) -> pd.DataFrame:
    # Check local cache first (refresh if older than 12 hours)
    if FORECAST_CACHE.exists():
        age_h = (time.time() - FORECAST_CACHE.stat().st_mtime) / 3600
        if age_h < 12:
            print(f"📦 Forecast cache is {age_h:.1f}h old — reusing local file")
            return pd.read_parquet(FORECAST_CACHE)
        else:
            print(f"📦 Forecast cache is {age_h:.1f}h old — refreshing from API")

    frames = []
    for name, (lat, lon) in tqdm(CITIES.items(), desc="Forecast"):
        df = fetch_forecast_one_city(name, lat, lon, forecast_days)
        if not df.empty:
            frames.append(df)
        time.sleep(CONFIG["polite_delay_s"])

    if not frames:
        # Fall back to stale cache if API failed entirely
        if FORECAST_CACHE.exists():
            print("⚠️  All forecast API calls failed — falling back to stale cache")
            return pd.read_parquet(FORECAST_CACHE)
        raise RuntimeError("No forecast data — API failed and no local cache exists.")

    return pd.concat(frames, ignore_index=True)


weather_fcst = fetch_all_forecasts(CONFIG["forecast_horizon_days"])
weather_fcst.to_parquet(FORECAST_CACHE, index=False)
print(f"✅ Forecast weather: {len(weather_fcst):,} rows · "
      f"horizon {CONFIG['forecast_horizon_days']} days")
weather_fcst.head()

📦 Forecast cache is 0.1h old — reusing local file
✅ Forecast weather: 6,144 rows · horizon 16 days


,Timestamp,Temperature_C,Humidity_percent,Rain_mm,Wind_Speed_kmh,Wind_Direction_deg,Pressure_hPa,Solar_Radiation_Wm2,Soil_Temp_C,Soil_Moisture,City,Latitude,Longitude
0,2026-04-26 00:00:00+00:00,7.455,90.0,0.0,4.896529,107.102814,973.329834,0.0,10.30,0.118,Ganja,40.6828,46.3606
1,2026-04-26 01:00:00+00:00,7.305,94.0,0.0,7.235910,84.289497,973.493835,0.0,9.90,0.118,Ganja,40.6828,46.3606
2,2026-04-26 02:00:00+00:00,7.055,97.0,0.0,6.162207,83.290260,973.640930,0.0,9.55,0.118,Ganja,40.6828,46.3606
3,2026-04-26 03:00:00+00:00,7.155,97.0,0.0,3.259938,83.659904,974.038513,17.0,9.45,0.118,Ganja,40.6828,46.3606
4,2026-04-26 04:00:00+00:00,7.455,95.0,0.0,1.484318,345.963715,974.280945,84.0,9.65,0.118,Ganja,40.6828,46.3606


## §4 · Fire labels — NASA FIRMS sensor archives

Place FIRMS sensor archive CSVs under `data/raw/firms/<sensor>/`.
The folder ships with archives for MODIS C6.1, SUOMI VIIRS C2, J1 VIIRS C2,
J2 VIIRS C2.

If the folder is empty (or missing entirely), the cell falls back to fire
labels stored in `data/raw/legacy/merged.parquet` so downstream notebooks
still run end-to-end.

In [7]:
def load_firms_from_disk(path: Path) -> pd.DataFrame:
    """Concatenate every CSV under <path>/<sensor>/ recursively."""
    if not path.exists():
        print(f"⚠️  FIRMS path missing: {path}")
        return pd.DataFrame()

    csvs = list(path.rglob("*.csv")) + list(path.rglob("*.CSV"))
    if not csvs:
        print(f"⚠️  No CSVs under {path}")
        return pd.DataFrame()

    print(f"📂 {len(csvs)} CSV files under {path}")
    frames = []
    for f in csvs:
        try:
            df = pd.read_csv(f, low_memory=False, on_bad_lines="skip")
            df["source_file"]   = f.name
            df["sensor_folder"] = f.parent.name
            frames.append(df)
        except Exception as e:
            print(f"   ⚠️  skipped {f.name}: {e}")

    out = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    print(f"✅ Loaded {len(out):,} fire detections")
    return out


fires_raw = load_firms_from_disk(FIRMS)

if not fires_raw.empty:
    fires_raw.columns = [c.strip().lower() for c in fires_raw.columns]
    fires_raw = fires_raw.rename(columns={"lat": "latitude", "lon": "longitude"})

    dedupe_cols = [c for c in ["latitude", "longitude", "acq_date",
                               "acq_time", "satellite"] if c in fires_raw.columns]
    if dedupe_cols:
        fires_raw = fires_raw.drop_duplicates(subset=dedupe_cols)

    # Type fixes for parquet
    if "confidence" in fires_raw.columns:
        fires_raw["confidence"] = fires_raw["confidence"].astype(str)
    if "version" in fires_raw.columns:
        fires_raw["version"] = fires_raw["version"].astype(str).fillna("")
    for col in ("brightness", "scan", "track", "frp"):
        if col in fires_raw.columns:
            fires_raw[col] = pd.to_numeric(fires_raw[col], errors="coerce")

    fires_raw = fires_raw.reset_index(drop=True)
    fires_raw.to_parquet(RAW / "firms_raw.parquet", index=False)
    print(f"💾 Saved firms_raw.parquet · sensors: {sorted(fires_raw['sensor_folder'].unique())}")
fires_raw.head() if not fires_raw.empty else "no fires loaded"

📂 6 CSV files under /home/manheim666/Desktop/ARIAN ALL FOLDERS/ARIAN_ASIF/data/raw/firms
✅ Loaded 141,493 fire detections
💾 Saved firms_raw.parquet · sensors: ['J1 VIIRS C2', 'MODIS C6.1', 'SUOMI VIIRS C2']


,latitude,longitude,brightness,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_t31,frp,daynight,type,source_file,sensor_folder
0,40.4183,47.4493,301.1,1.3,1.1,2012-02-20,941,Aqua,MODIS,44,6.03,286.0,6.9,D,0.0,fire_archive_M-C61_743125.csv,MODIS C6.1
1,39.2516,46.7231,315.8,1.1,1.1,2012-02-23,1011,Aqua,MODIS,76,6.03,290.9,15.3,D,0.0,fire_archive_M-C61_743125.csv,MODIS C6.1
2,39.2500,46.7103,309.5,1.1,1.1,2012-02-23,1011,Aqua,MODIS,69,6.03,289.0,9.7,D,0.0,fire_archive_M-C61_743125.csv,MODIS C6.1
3,40.3432,47.2605,305.7,1.2,1.1,2012-02-23,1012,Aqua,MODIS,62,6.03,288.4,8.6,D,0.0,fire_archive_M-C61_743125.csv,MODIS C6.1
4,40.5906,47.4714,303.2,1.2,1.1,2012-02-23,1012,Aqua,MODIS,55,6.03,285.7,7.1,D,0.0,fire_archive_M-C61_743125.csv,MODIS C6.1


### Normalising fire detections to a daily binary label per city

In [8]:
# Approximate geo helpers (good for ~20 km buffer at 40°N)
_KM_PER_DEG_LAT = 110.574
def km_to_deg_lat(km: float) -> float: return km / _KM_PER_DEG_LAT
def km_to_deg_lon(km: float, lat: float) -> float:
    return km / (111.32 * np.cos(np.radians(lat)))

def bbox_around(lat: float, lon: float, buffer_km: float):
    dlat, dlon = km_to_deg_lat(buffer_km), km_to_deg_lon(buffer_km, lat)
    return lon - dlon, lat - dlat, lon + dlon, lat + dlat


def normalise_fires(fires_raw: pd.DataFrame) -> pd.DataFrame:
    cols = ["City", "Date", "Fire_Occurred", "fire_count", "mean_brightness", "max_frp"]
    if fires_raw.empty:
        return pd.DataFrame(columns=cols)

    df = fires_raw.copy()
    df["Date"] = pd.to_datetime(df["acq_date"]).dt.date

    bboxes = {c: bbox_around(lat, lon, CONFIG["fire_buffer_km"])
              for c, (lat, lon) in CITIES.items()}

    def find_city(lat: float, lon: float):
        for c, (w, s, e, n) in bboxes.items():
            if s <= lat <= n and w <= lon <= e:
                return c
        return None

    df["City"] = [find_city(la, lo) for la, lo in zip(df["latitude"], df["longitude"])]
    df = df.dropna(subset=["City"])
    if df.empty:
        return pd.DataFrame(columns=cols)

    df["brightness_norm"] = df.get("bright_ti4",
                                   df.get("brightness", pd.Series(np.nan, index=df.index)))
    df["frp"] = df.get("frp", pd.Series(0.0, index=df.index))

    daily = (df.groupby(["City", "Date"])
               .agg(fire_count=("latitude", "count"),
                    mean_brightness=("brightness_norm", "mean"),
                    max_frp=("frp", "max"))
               .reset_index())
    daily["Fire_Occurred"] = 1
    return daily


fires_daily = normalise_fires(fires_raw)

if fires_daily.empty:
    # Fall back to legacy merged.csv so the pipeline still runs end-to-end
    legacy_csv_path = LEGACY / "merged.csv"
    legacy_pq_path  = LEGACY / "merged.parquet"

    if legacy_csv_path.exists():
        print("ℹ️  Backfilling fire labels from legacy merged.csv")
        try:
            legacy = pd.read_csv(legacy_csv_path, low_memory=False,
                                 usecols=["City", "Timestamp", "Fire_Occurred",
                                          "Burned_Area_hectares"])
            legacy["Timestamp"] = pd.to_datetime(legacy["Timestamp"])
            legacy["Date"] = legacy["Timestamp"].dt.date
            fires_daily = (legacy.groupby(["City", "Date"])
                           .agg(Fire_Occurred=("Fire_Occurred", "max"))
                           .reset_index())
            for col in ("fire_count", "mean_brightness", "max_frp"):
                fires_daily[col] = np.nan
        except Exception as exc:
            print(f"   ⚠️  merged.csv read failed: {exc}")
    elif legacy_pq_path.exists():
        print("ℹ️  Backfilling fire labels from legacy merged.parquet")
        legacy = pd.read_parquet(legacy_pq_path,
                                 columns=["City", "Timestamp", "Fire_Occurred",
                                          "Burned_Area_hectares"])
        legacy["Date"] = legacy["Timestamp"].dt.date
        fires_daily = (legacy.groupby(["City", "Date"])
                       .agg(Fire_Occurred=("Fire_Occurred", "max"))
                       .reset_index())
        for col in ("fire_count", "mean_brightness", "max_frp"):
            fires_daily[col] = np.nan
    else:
        print("⚠️  No FIRMS data and no legacy fallback — Fire_Occurred will be 0.")

if not fires_daily.empty:
    fires_daily["Date"] = pd.to_datetime(fires_daily["Date"])
    fires_daily.to_parquet(PROCESSED / "fires_daily.parquet", index=False)
    print(f"✅ Daily fire labels: {len(fires_daily):,} fire-days · "
          f"{fires_daily['City'].nunique()} cities")
fires_daily.head()

✅ Daily fire labels: 9,059 fire-days · 16 cities


,City,Date,fire_count,mean_brightness,max_frp,Fire_Occurred
0,Baku,2012-03-28,1,300.8400,1.30,1
1,Baku,2012-03-29,1,295.6900,0.61,1
2,Baku,2012-03-31,4,328.5525,5.02,1
3,Baku,2012-04-02,1,333.2400,16.32,1
4,Baku,2012-04-12,2,340.6250,7.45,1


## §5 · Static geography (one-time)

In [9]:
def fetch_elevation(lat: float, lon: float) -> float:
    url = f"https://api.open-elevation.com/api/v1/lookup?locations={lat},{lon}"
    try:
        r = retry_session.get(url, timeout=20)
        return float(r.json()["results"][0]["elevation"])
    except Exception as exc:
        print(f"   ⚠️  elevation failed for ({lat:.3f},{lon:.3f}): {exc}")
        return np.nan


def fetch_slope_proxy(lat: float, lon: float, step_km: float = 1.0) -> float:
    """Mean absolute elevation gradient over a 4-neighbour 1-km cross."""
    dlat, dlon = km_to_deg_lat(step_km), km_to_deg_lon(step_km, lat)
    pts = [(lat, lon), (lat+dlat, lon), (lat-dlat, lon),
           (lat, lon+dlon), (lat, lon-dlon)]
    locs = "|".join(f"{a},{b}" for a, b in pts)
    try:
        r = retry_session.get(
            f"https://api.open-elevation.com/api/v1/lookup?locations={locs}", timeout=30)
        c, n, s, e, w = [pt["elevation"] for pt in r.json()["results"]]
        grad_ns = abs(n - s) / (2 * step_km * 1000)
        grad_ew = abs(e - w) / (2 * step_km * 1000)
        return float(np.degrees(np.arctan(np.hypot(grad_ns, grad_ew))))
    except Exception as exc:
        print(f"   ⚠️  slope failed for ({lat:.3f},{lon:.3f}): {exc}")
        return np.nan


STATIC_PATH = REFERENCE / "static_geography.parquet"
if STATIC_PATH.exists():
    print(f"ℹ️  Reusing cached static geography → {STATIC_PATH}")
    static_df = pd.read_parquet(STATIC_PATH)
else:
    rows = []
    for name, (lat, lon) in tqdm(CITIES.items(), desc="Geography"):
        rows.append({
            "City": name, "Latitude": lat, "Longitude": lon,
            "Elevation": fetch_elevation(lat, lon),
            "Slope":     fetch_slope_proxy(lat, lon),
        })
        time.sleep(0.5)
    static_df = pd.DataFrame(rows)

    # Enrich with land-cover / population from legacy CSV if available
    legacy_static = LEGACY / "all_cities_static_geography.csv"
    if legacy_static.exists():
        extra = pd.read_csv(legacy_static)
        keep_cols = [c for c in ["City", "Trees_pct", "Urban_pct", "Pop_Total"]
                     if c in extra.columns]
        if len(keep_cols) > 1:
            static_df = static_df.merge(extra[keep_cols], on="City", how="left")

    static_df.to_parquet(STATIC_PATH, index=False)

static_df

ℹ️  Reusing cached static geography → /home/manheim666/Desktop/ARIAN ALL FOLDERS/ARIAN_ASIF/data/reference/static_geography.parquet


,City,Latitude,Longitude,Elevation,Slope,Trees_pct,Urban_pct,Pop_Total
0,Ganja,40.6828,46.3606,417.0,1.255740,6.375932,10.206253,4.014443e+05
1,Baku,40.4093,49.8671,24.0,0.405136,1.950468,34.169409,2.048268e+06
2,Zaqatala,41.6296,46.6433,500.0,2.152348,68.954934,1.625946,1.109243e+05
3,Lankaran,38.7523,48.8475,-20.0,0.081028,27.364711,3.866356,2.240667e+05
4,Shaki,41.1975,47.1694,577.0,3.696211,33.073795,3.331254,1.115553e+05
5,Jalilabad,39.2089,48.2986,242.0,4.657300,8.951281,2.751703,1.338332e+05
6,Sumqayit,40.5897,49.6686,-12.0,0.362365,1.006923,15.498425,7.755712e+05
7,Mingachevir,40.7639,47.0595,26.0,0.831224,2.670386,3.534099,1.732764e+05
8,Shirvan,39.9317,48.9299,-7.0,1.939295,1.480768,5.821671,1.510896e+05
9,Quba,41.3611,48.5261,575.0,2.176945,45.866631,3.944225,1.342683e+05


## §6 · Build the daily training matrix

In [10]:
def hourly_to_daily(df: pd.DataFrame) -> pd.DataFrame:
    """Aggregate hourly weather to daily, preserving fire-relevant statistics."""
    df = df.copy()
    df["Date"] = df["Timestamp"].dt.date

    agg = (df.groupby(["City", "Date"])
             .agg(Temperature_C_mean    =("Temperature_C", "mean"),
                  Temperature_C_max     =("Temperature_C", "max"),
                  Temperature_C_min     =("Temperature_C", "min"),
                  Humidity_percent_mean =("Humidity_percent", "mean"),
                  Humidity_percent_min  =("Humidity_percent", "min"),
                  Wind_Speed_kmh_mean   =("Wind_Speed_kmh", "mean"),
                  Wind_Speed_kmh_max    =("Wind_Speed_kmh", "max"),
                  Rain_mm_sum           =("Rain_mm", "sum"),
                  Pressure_hPa_mean     =("Pressure_hPa", "mean"),
                  Solar_Radiation_Wm2_mean=("Solar_Radiation_Wm2", "mean"),
                  Soil_Temp_C_mean      =("Soil_Temp_C", "mean"),
                  Soil_Moisture_mean    =("Soil_Moisture", "mean"))
             .reset_index())
    agg["Date"] = pd.to_datetime(agg["Date"])
    return agg


daily = hourly_to_daily(weather_hist)
print(f"Daily weather matrix: {len(daily):,} rows")

# ── Merge fire labels ──────────────────────────────────────────────────────
if not fires_daily.empty:
    daily = daily.merge(fires_daily, on=["City", "Date"], how="left")

for c in ("Fire_Occurred", "fire_count"):
    if c in daily.columns:
        daily[c] = daily[c].fillna(0)
if "Fire_Occurred" not in daily.columns:
    daily["Fire_Occurred"] = 0

# ── Merge static geography ─────────────────────────────────────────────────
daily = daily.merge(static_df, on="City", how="left")

# ── Calendar features ──────────────────────────────────────────────────────
daily["Year"]      = daily["Date"].dt.year
daily["Month"]     = daily["Date"].dt.month
daily["DayOfYear"] = daily["Date"].dt.dayofyear
daily["DayOfWeek"] = daily["Date"].dt.dayofweek

daily = daily.sort_values(["City", "Date"]).reset_index(drop=True)
daily.to_parquet(PROCESSED / "master_daily.parquet", index=False)

print(f"✅ Master daily matrix → {PROCESSED / 'master_daily.parquet'}")
print(f"   Shape: {daily.shape}")
print(f"   Date range: {daily['Date'].min().date()} → {daily['Date'].max().date()}")
print(f"   Fire-days: {int(daily['Fire_Occurred'].sum())} "
      f"({daily['Fire_Occurred'].mean()*100:.2f} %)")
daily.head()

Daily weather matrix: 83,408 rows
✅ Master daily matrix → /home/manheim666/Desktop/ARIAN ALL FOLDERS/ARIAN_ASIF/data/processed/master_daily.parquet
   Shape: (83408, 29)
   Date range: 2012-01-19 → 2026-04-27
   Fire-days: 9059 (10.86 %)


,City,Date,Temperature_C_mean,Temperature_C_max,Temperature_C_min,Humidity_percent_mean,Humidity_percent_min,Wind_Speed_kmh_mean,Wind_Speed_kmh_max,Rain_mm_sum,...,Longitude,Elevation,Slope,Trees_pct,Urban_pct,Pop_Total,Year,Month,DayOfYear,DayOfWeek
0,Baku,2012-01-19,3.961500,4.024,3.874,81.971840,81.076520,38.260614,40.603470,0.4,...,49.8671,24.0,0.405136,1.950468,34.169409,2.048268e+06,2012,1,19,3
1,Baku,2012-01-20,2.924000,3.924,2.124,83.210192,79.108710,47.359066,51.751170,11.4,...,49.8671,24.0,0.405136,1.950468,34.169409,2.048268e+06,2012,1,20,4
2,Baku,2012-01-21,1.913583,2.624,1.024,73.228824,70.361880,25.123994,43.274933,1.5,...,49.8671,24.0,0.405136,1.950468,34.169409,2.048268e+06,2012,1,21,5
3,Baku,2012-01-22,2.146917,2.774,1.324,73.143316,68.247560,16.954147,20.696087,0.3,...,49.8671,24.0,0.405136,1.950468,34.169409,2.048268e+06,2012,1,22,6
4,Baku,2012-01-23,1.284417,1.624,0.874,75.765419,70.665695,17.078632,19.881649,2.8,...,49.8671,24.0,0.405136,1.950468,34.169409,2.048268e+06,2012,1,23,0


## §7 · Sanity-check map

In [11]:
import folium

m = folium.Map(location=[40.5, 47.5], zoom_start=7, tiles="cartodbpositron")
for name, (lat, lon) in CITIES.items():
    folium.CircleMarker(
        [lat, lon], radius=6, color="#c0392b", fill=True, fill_opacity=0.85,
        popup=f"<b>{name}</b><br>{lat:.3f}, {lon:.3f}",
    ).add_to(m)
    w, s, e, n = bbox_around(lat, lon, CONFIG["fire_buffer_km"])
    folium.Rectangle(bounds=[(s, w), (n, e)],
                     color="#e67e22", weight=1, fill=False).add_to(m)

m.save(str(OUTPUTS / "phase1_ingest_check.html"))
print(f"💾 Saved → {OUTPUTS / 'phase1_ingest_check.html'}")
m

💾 Saved → /home/manheim666/Desktop/ARIAN ALL FOLDERS/ARIAN_ASIF/outputs/phase1_ingest_check.html


---
**Next:** open `02_Weather_Forecasting.ipynb` to train the weather models.